# CIC-IDS2017 전처리 파이프라인 v2
**캡스톤디자인 I** - 특징 추출 최적화 기반 암호화된 공격 트래픽 탐지 및 분류 시스템  
팀장: 김강민 | 팀원: 서은빈

---
### 노이즈 제거 목록
| 단계 | 처리 내용 |
|---|---|
| 1 | Label 인코딩 오류 수정 |
| 2 | Flow Duration ≤ 0 제거 (단일 패킷 / 버그) |
| 3 | Inf 값 포함 행 제거 |
| 4 | NaN 포함 행 제거 |
| 5 | 소수 클래스 제거 (Infiltration, Heartbleed, Sql Injection) |
| 6 | 음수값 행 제거 (CICFlowMeter 버그) |
| 7 | 완전 중복 행 제거 |
| 8 | Init_Win_bytes -1 → 0 대체 (UDP 플로우 마커) |
| 9 | 분산=0 상수 컬럼 제거 |
| 10 | SII 컬럼 제거 (Destination Port) |
| 11 | 버그 컬럼 제거 (Fwd Header Length.1) |

### Google Drive 폴더 구조
```
Google Drive/
└── capstone2026/
    ├── MachineLearningCSV/
    │   └── MachineLearningCVE/   ← CSV 파일 8개
    └── output/                   ← 전처리 결과 저장
```

## 0. Google Drive 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR   = '/content/drive/MyDrive/capstone2026'
DATA_DIR   = os.path.join(BASE_DIR, 'MachineLearningCSV/MachineLearningCVE/*.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'BASE_DIR  : {BASE_DIR}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
BASE_DIR  : /content/drive/MyDrive/capstone2026
DATA_DIR  : /content/drive/MyDrive/capstone2026/MachineLearningCSV/MachineLearningCVE/*.csv
OUTPUT_DIR: /content/drive/MyDrive/capstone2026/output


## 1. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import glob

print('pandas:', pd.__version__)
print('numpy :', np.__version__)

pandas: 2.2.2
numpy : 2.0.2


## 2. 설정값

In [ ]:
# ── 소수 클래스 (학습 불가) ──
EXCLUDE_LABELS = [
    'Infiltration',
    'Heartbleed',
    'Web Attack \ufffd Sql Injection',
    'Web Attack - Sql Injection',
]

# ── Label 인코딩 오류 수정 ──
LABEL_FIX = {
    'Web Attack \ufffd Brute Force': 'Web Attack - Brute Force',
    'Web Attack \ufffd XSS':         'Web Attack - XSS',
    'Web Attack \ufffd Sql Injection': 'Web Attack - Sql Injection',
}

# ── SII 컬럼 (숏컷 러닝 방지) ──
SII_COLUMNS = ['Destination Port']

# ── 버그/중복 컬럼 ──
BUG_COLUMNS = ['Fwd Header Length.1']

# ── 음수 불가 컬럼 (음수 = CICFlowMeter 버그) ──
# Init_Win_bytes 제외 (-1은 UDP 마커라 별도 처리)
NEGATIVE_INVALID_COLS = [
    'Flow Duration',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Flow IAT Mean',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Min',
    'Fwd Header Length',
    'min_seg_size_forward',
]

# ── Init_Win_bytes: -1 → 0 대체 ──
INIT_WIN_COLUMNS = [
    'Init_Win_bytes_forward',
    'Init_Win_bytes_backward',
]

print('설정 완료')

설정 완료


## 3. 데이터 로드

In [ ]:
def load_data(data_dir):
    files = sorted(glob.glob(data_dir))
    print(f'[로드] {len(files)}개 파일 발견')
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, encoding='utf-8', low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(f, encoding='latin-1', low_memory=False)
        df.columns = df.columns.str.strip()
        for col in df.select_dtypes(include=['float64']).columns:
            df[col] = df[col].astype('float32')
        for col in df.select_dtypes(include=['int64']).columns:
            df[col] = df[col].astype('int32')
        print(f'  {os.path.basename(f)}: {len(df):,}행')
        dfs.append(df)
    all_df = pd.concat(dfs, ignore_index=True)
    print(f'\n[로드 완료] 총 {len(all_df):,}행 | {len(all_df.columns)}컬럼')
    return all_df

df = load_data(DATA_DIR)

[로드] 8개 파일 발견
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745행
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467행
  Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033행
  Monday-WorkingHours.pcap_ISCX.csv: 529,918행
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602행
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366행
  Tuesday-WorkingHours.pcap_ISCX.csv: 445,909행
  Wednesday-workingHours.pcap_ISCX.csv: 692,703행

[로드 완료] 총 2,830,743행 | 79컬럼


## 4. 노이즈 제거

In [ ]:
def remove_noise(df):
    original_len = len(df)
    log = []

    # 1. Label 인코딩 오류 수정
    df['Label'] = df['Label'].replace(LABEL_FIX)
    log.append('Label 인코딩 오류 수정 완료')

    # 2. Flow Duration <= 0 제거
    before = len(df)
    df = df[df['Flow Duration'] > 0]
    log.append(f'Flow Duration <= 0 제거: {before - len(df):,}행')

    # 3. Inf 값 포함 행 제거
    before = len(df)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df = df[~np.isinf(df[num_cols]).any(axis=1)]
    log.append(f'Inf 값 포함 행 제거: {before - len(df):,}행')

    # 4. NaN 제거
    before = len(df)
    df = df.dropna()
    log.append(f'NaN 포함 행 제거: {before - len(df):,}행')

    # 5. 소수 클래스 제거
    before = len(df)
    df = df[~df['Label'].isin(EXCLUDE_LABELS)]
    log.append(f'소수 클래스 제거: {before - len(df):,}행')

    # 6. 음수값 행 제거 (CICFlowMeter 버그)
    before = len(df)
    neg_cols = [c for c in NEGATIVE_INVALID_COLS if c in df.columns]
    df = df[~(df[neg_cols] < 0).any(axis=1)]
    log.append(f'음수값 행 제거 (버그): {before - len(df):,}행')

    # 7. 완전 중복 행 제거 스킵
    # float32 변환으로 인한 정밀도 손실로 false positive 발생
    # (다른 플로우가 같아 보이는 현상) → 실제 중복 아님

    # 8. Init_Win_bytes -1 → 0 대체 (UDP 플로우 마커)
    win_cols = [c for c in INIT_WIN_COLUMNS if c in df.columns]
    for col in win_cols:
        df[col] = df[col].replace(-1, 0)
    log.append(f'Init_Win_bytes -1 → 0 대체')

    print('[노이즈 제거]')
    for l in log:
        print(f'  ✓ {l}')
    print(f'  → {original_len:,}행 → {len(df):,}행 (제거: {original_len - len(df):,}행)')
    return df.reset_index(drop=True)

df = remove_noise(df)

[노이즈 제거]
  ✓ Label 인코딩 오류 수정 완료
  ✓ Flow Duration <= 0 제거: 2,982행
  ✓ Inf 값 포함 행 제거: 0행
  ✓ NaN 포함 행 제거: 0행
  ✓ 소수 클래스 제거: 68행
  ✓ 음수값 행 제거 (버그): 2,805행
  ✓ Init_Win_bytes -1 → 0 대체
  → 2,830,743행 → 2,824,888행 (제거: 5,855행)


## 5. 컬럼 제거 (SII + 버그 + 분산=0)

In [ ]:
def remove_columns(df):
    print('[컬럼 제거]')

    # 분산=0 컬럼 실제 데이터 기반으로 재확인
    num_cols = df.select_dtypes(include=[np.number]).columns
    zero_var_cols = [c for c in num_cols if df[c].std() == 0]

    all_drop = list(set(SII_COLUMNS + BUG_COLUMNS + zero_var_cols))
    to_drop  = [c for c in all_drop if c in df.columns]

    for c in sorted(to_drop):
        if c in SII_COLUMNS:
            tag = 'SII  '
        elif c in BUG_COLUMNS:
            tag = '버그  '
        else:
            tag = '분산=0'
        print(f'  ✓ [{tag}] "{c}" 제거')

    df = df.drop(columns=to_drop)
    print(f'  → 잔여 특징 수: {len(df.columns) - 1}개 (Label 제외)')
    return df

df = remove_columns(df)

[컬럼 제거]
  ✓ [분산=0] "Bwd Avg Bulk Rate" 제거
  ✓ [분산=0] "Bwd Avg Bytes/Bulk" 제거
  ✓ [분산=0] "Bwd Avg Packets/Bulk" 제거
  ✓ [분산=0] "Bwd PSH Flags" 제거
  ✓ [분산=0] "Bwd URG Flags" 제거
  ✓ [SII  ] "Destination Port" 제거
  ✓ [분산=0] "Fwd Avg Bulk Rate" 제거
  ✓ [분산=0] "Fwd Avg Bytes/Bulk" 제거
  ✓ [분산=0] "Fwd Avg Packets/Bulk" 제거
  ✓ [버그  ] "Fwd Header Length.1" 제거
  → 잔여 특징 수: 68개 (Label 제외)


## 6. 최종 검증

In [ ]:
print('[최종 검증]')
num_cols   = df.select_dtypes(include=[np.number]).columns
inf_remain = int(np.isinf(df[num_cols]).sum().sum())
nan_remain = int(df.isnull().sum().sum())
neg_remain = int((df[num_cols] < 0).sum().sum())
sii_remain = [c for c in SII_COLUMNS if c in df.columns]
zv_remain  = [c for c in num_cols if df[c].std() == 0]

print(f'  Inf 잔여     : {inf_remain}개  {"✅" if inf_remain == 0 else "❌"}')
print(f'  NaN 잔여     : {nan_remain}개  {"✅" if nan_remain == 0 else "❌"}')
print(f'  음수값 잔여  : {neg_remain}개  {"✅" if neg_remain == 0 else "❌"}')
print(f'  SII 컬럼 잔여: {sii_remain if sii_remain else "없음"}  {"✅" if not sii_remain else "❌"}')
print(f'  분산=0 컬럼  : {zv_remain if zv_remain else "없음"}  {"✅" if not zv_remain else "❌"}')

print(f'\n[최종 데이터셋]')
print(f'  행 수  : {len(df):,}')
print(f'  특징 수: {len(df.columns) - 1}개 (Label 제외)')

print('\n[최종 클래스 분포]')
vc = df['Label'].value_counts()
total = len(df)
for k, v in vc.items():
    print(f'  {k}: {v:,} ({v/total*100:.2f}%)')

[최종 검증]
  Inf 잔여     : 0개  ✅
  NaN 잔여     : 0개  ✅
  음수값 잔여  : 0개  ✅
  SII 컬럼 잔여: 없음  ✅
  분산=0 컬럼  : 없음  ✅

[최종 데이터셋]
  행 수  : 2,824,888
  특징 수: 68개 (Label 제외)

[최종 클래스 분포]
  BENIGN: 2,268,589 (80.31%)
  DoS Hulk: 229,965 (8.14%)
  PortScan: 158,804 (5.62%)
  DDoS: 128,006 (4.53%)
  DoS GoldenEye: 10,288 (0.36%)
  FTP-Patator: 7,931 (0.28%)
  SSH-Patator: 5,895 (0.21%)
  DoS slowloris: 5,796 (0.21%)
  DoS Slowhttptest: 5,499 (0.19%)
  Bot: 1,956 (0.07%)
  Web Attack - Brute Force: 1,507 (0.05%)
  Web Attack - XSS: 652 (0.02%)


## 7. 저장

In [ ]:
out_path = os.path.join(OUTPUT_DIR, 'cicids2017_cleaned_v2.csv')
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'[저장 완료] {out_path}')
print(f'  최종: {len(df):,}행 | 특징: {len(df.columns)-1}개')

[저장 완료] /content/drive/MyDrive/capstone2026/output/cicids2017_cleaned_v2.csv
  최종: 2,824,888행 | 특징: 68개
